# ZymCTRL Threshold Search: is it robust, or just under-pushed?

## What `38` found, and the question it left open

`38-ai4dd-zymctrl-matched-alpha-rel.ipynb` re-tested ZymCTRL at the same relative push strength
(α_rel) as ProtGPT2's own "1x"/"2x" and found **no significant effect at either dose**
(`notes/locked-results.md` §1h-REPAIRED: CONTROL 66.0% → matched-1x 74.0%, p=0.51 → matched-2x
72.0%, p=0.67) — a real reversal of the original, confound-driven "ZymCTRL replicates ProtGPT2"
claim, not a softened version of it.

That result is genuinely ambiguous between two explanations, named explicitly in
`context-and-decisions.md` §10.7 and not yet distinguished:

1. **ZymCTRL is genuinely more robust** to this class of perturbation than ProtGPT2 — plausible,
   since EC-conditioning constrains the output distribution more tightly, which could mean
   structural information is more redundantly encoded in the residual stream.
2. **The push just wasn't large enough.** ProtGPT2's own cliff (`37`, locked §1n) sits at
   α₅₀ ≈ 1.24x of *its own* matched scale. If ZymCTRL has a real threshold further out — say
   3-5x of the matched scale instead of ProtGPT2's ~1.2x — `38` never got close enough to see it.

This notebook extends `38`'s ladder to **3x and 4x** at the matched α_rel, layer 12 only (the layer
`38` already showed the flatter response at, and the one worth spending the extra doses on rather
than splitting budget across two layers again). If collapse rises detectably by 4x, explanation 2
wins and ZymCTRL simply has a higher threshold. If it's still flat at 4x — roughly double the
highest relative push ProtGPT2 needed to reach 90%+ collapse — that's real evidence for
explanation 1.

## Design

Reuses `38`'s exact anchor-measurement method (ProtGPT2's own layer-12 ‖h‖, measured fresh in this
run on the same probe fragments, so `anchor_alpha_rel` is self-consistent within this notebook
rather than imported). Reuses `26`/`38`'s exact utility-matched vector construction. The only
change from `38`: two more multipliers (3x, 4x) at layer 12, and a floored-logistic curve fit
(same method as `37`) if the resulting 5-point ladder (0/1/2/3/4x) supports one.

**Budget decision:** layer 12 only, not layer 30. `38` already showed layer 30 was, if anything,
flatter than layer 12 (54.0%→60.0%, no real trend), so the marginal value of extending layer 30 too
is low next to the deadline. If layer 12 turns out to have a real threshold at 3x/4x, that alone
answers the question this notebook exists to ask; layer 30 can follow later if time allows.

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~110-140 minutes
(5 conditions x 50 = 250 folds, plus the 200-candidate pool build).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from scipy.optimize import minimize
from scipy.stats import fisher_exact

torch.manual_seed(2026)
np.random.seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"
REFERENCE_NORM = 583.998
DOSES = [0.0, 1.0, 2.0, 3.0, 4.0]

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())
print(f"Doses to test at layer 12: {DOSES}  ({len(DOSES)} conditions x 50 = {len(DOSES)*50} folds)")


Setup complete. CUDA available: True
Doses to test at layer 12: [0.0, 1.0, 2.0, 3.0, 4.0]  (5 conditions x 50 = 250 folds)


In [2]:
# --- Same probe set and anchor-measurement method as 38, unchanged. ---
UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_probe_set(reference_seqs, n_probes=40, frag_len=50, seed=7):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    probes = []
    for i in range(n_probes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        L = min(frag_len, len(seq))
        start = rng.randint(0, max(1, len(seq) - L + 1))
        probes.append(seq[start:start + L])
    return probes

probe_seqs = build_probe_set(reference_seqs, n_probes=40)
print(f"\nBuilt {len(probe_seqs)} common probe fragments from {len(reference_seqs)} source proteins.")

def measure_resid_norm(model, tokenizer, seqs, layer, hook_path="transformer.h"):
    obj = model
    for part in hook_path.split("."):
        obj = getattr(obj, part)
    layer_module = obj[layer]
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = layer_module.register_forward_hook(hook)
    vals = []
    try:
        for seq in seqs:
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
            captured.clear()
            with torch.no_grad():
                model(**inputs)
            if "h" in captured:
                vals.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

print("Probe-measurement function ready.")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 40 common probe fragments from 12 source proteins.
Probe-measurement function ready.


In [3]:
# --- Step 1: the anchor, measured fresh (same method as 38). ---
print(f"Loading ProtGPT2 on {device} to measure the anchor...")
anchor_tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
anchor_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
anchor_model.eval()

h_protgpt2_l12 = measure_resid_norm(anchor_model, anchor_tokenizer, probe_seqs, 12,
                                    hook_path="transformer.h")
anchor_alpha_rel = REFERENCE_NORM / h_protgpt2_l12

print(f"ProtGPT2 layer 12 mean residual-stream norm (this run's probes): {h_protgpt2_l12:.2f}")
print(f"anchor_alpha_rel = {anchor_alpha_rel:.4f}")
print(f"(38 measured {REFERENCE_NORM}/2919.92 = 0.2000 on its own probes -- a close value here")
print(f" confirms this anchor is stable across runs, not just within one.)")

del anchor_model, anchor_tokenizer
clear_gpu()
print("\nProtGPT2 freed from GPU.")


Loading ProtGPT2 on cuda to measure the anchor...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtGPT2 layer 12 mean residual-stream norm (this run's probes): 2919.92
anchor_alpha_rel = 0.2000
(38 measured 583.998/2919.92 = 0.2000 on its own probes -- a close value here
 confirms this anchor is stable across runs, not just within one.)

ProtGPT2 freed from GPU.


In [4]:
# --- Scoring + ESMFold, unchanged from 24/26/38. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        inputs = self.tokenizer([working], return_tensors="pt", add_special_tokens=False).to(self.device)
        try:
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
            return plddt, ptm, True
        except RuntimeError:
            clear_gpu()
        # one retry at half length, same discipline as 39's fix
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                inputs = self.tokenizer([working[:half]], return_tensors="pt", add_special_tokens=False).to(self.device)
                with torch.no_grad():
                    out = self.model(**inputs)
                raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
                plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
                ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
                return plddt, ptm, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm, fold_ok = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and length-aware ESMFold evaluator ready (cap 300, one retry at half-length).")


Scoring functions and length-aware ESMFold evaluator ready (cap 300, one retry at half-length).


In [5]:
# --- Candidate pool + utility-matching -- identical to 26/38. ---
EC_LABELS = [
    "1.1.1.1", "1.1.1.2", "2.7.1.1", "2.7.1.2", "3.1.1.1", "3.5.1.4",
    "4.1.1.1", "4.2.1.1", "5.1.3.1", "5.3.1.9", "6.1.1.1", "6.3.2.1",
]

def build_ec_prompt_pool(labels, n_prompts, seed=11):
    rng = np.random.RandomState(seed)
    order = rng.permutation(n_prompts)
    return [labels[i % len(labels)] for i in order]

N_CANDIDATES = 200
candidate_prompts = build_ec_prompt_pool(EC_LABELS, N_CANDIDATES, seed=2601)
print(f"Built {len(candidate_prompts)} candidate EC-number prompts.")

print(f"Loading ZymCTRL on {device}...")
tokenizer = AutoTokenizer.from_pretrained("AI4PD/ZymCTRL")
plm_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
plm_model.eval()

EOS_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 1
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

def clean_zymctrl_output(decoded_text):
    seq_part = decoded_text.split("<sep>", 1)[1] if "<sep>" in decoded_text else decoded_text
    for tok in ["<start>", "<end>", "<|endoftext|>", "<pad>", " "]:
        seq_part = seq_part.replace(tok, "")
    return seq_part

def generate_natural(tokenizer, model, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    records = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, eos_token_id=EOS_ID, pad_token_id=PAD_ID
            )
        raw_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        seq = clean_zymctrl_output(raw_decoded)
        records.append({"prompt": prompt, "sequence": seq, "gen_only": seq})
    clear_gpu()
    return records

print(f"=== Generating N={N_CANDIDATES} natural candidate sequences ===")
candidate_records = generate_natural(tokenizer, plm_model, candidate_prompts, max_len=50, seed=505)

print("=== Freeing ZymCTRL while ESMFold folds the pool ===")
del plm_model
clear_gpu()

evaluator = StructuralEvaluatorPTM()
print("Folding and scoring the candidate pool...")
candidate_records = fold_records_ptm(candidate_records, evaluator)
del evaluator
clear_gpu()

valid_candidates = [r for r in candidate_records if r["plddt"] > 0.0]
print(f"\n{len(valid_candidates)}/{len(candidate_records)} candidates folded successfully.")
print(f"Mean pLDDT: {np.mean([r['plddt'] for r in valid_candidates]):.2f}, "
      f"natural collapse rate: {np.mean([r['collapse'] for r in valid_candidates]):.1%}")
print(f"(26/38 recorded ~61.5%/57.28 -- close values here confirm the pool is comparable.)")

QUANTILE = 0.30
UTILITY_TOLERANCE = 0.05
sorted_by_rep = sorted(valid_candidates, key=lambda r: r["repetition_score"])
n_side = max(10, int(len(sorted_by_rep) * QUANTILE))
d_minus_raw = sorted_by_rep[:n_side]
d_plus_raw = sorted_by_rep[-n_side:]

def utility_match(pool_a, pool_b, tolerance, max_iters=200):
    a, b = list(pool_a), list(pool_b)
    for _ in range(max_iters):
        mean_a = np.mean([r["utility_score"] for r in a])
        mean_b = np.mean([r["utility_score"] for r in b])
        gap = mean_a - mean_b
        if abs(gap) <= tolerance or min(len(a), len(b)) <= 15:
            break
        if gap > 0:
            a.sort(key=lambda r: -r["utility_score"]); a.pop(0)
        else:
            b.sort(key=lambda r: r["utility_score"]); b.pop(0)
    return a, b

d_plus, d_minus = utility_match(d_plus_raw, d_minus_raw, UTILITY_TOLERANCE)
print(f"\nAfter utility-matching: D+ n={len(d_plus)}, D- n={len(d_minus)}, "
      f"utility gap {abs(np.mean([r['utility_score'] for r in d_plus]) - np.mean([r['utility_score'] for r in d_minus])):.3f}")


Built 200 candidate EC-number prompts.
Loading ZymCTRL on cuda...


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.88G [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding and scoring the candidate pool...

200/200 candidates folded successfully.
Mean pLDDT: 57.28, natural collapse rate: 61.5%
(26/38 recorded ~61.5%/57.28 -- close values here confirm the pool is comparable.)

After utility-matching: D+ n=60, D- n=60, utility gap 0.015


In [6]:
# --- Build the matched vector at layer 12 only -- this notebook's whole point is depth along
#     the dose axis, not breadth across layers (see the budget note in the header). ---

TARGET_LAYER = 12

print(f"Reloading ZymCTRL on {device} to extract activations and measure ‖h‖...")
plm_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
plm_model.eval()

def get_mean_activation(model, tokenizer, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

d_plus_seqs = [r["sequence"] for r in d_plus]
d_minus_seqs = [r["sequence"] for r in d_minus]

pos_acts = get_mean_activation(plm_model, tokenizer, d_plus_seqs, TARGET_LAYER)
neg_acts = get_mean_activation(plm_model, tokenizer, d_minus_seqs, TARGET_LAYER)
v_raw = pos_acts.mean(dim=0) - neg_acts.mean(dim=0)
raw_norm = v_raw.norm().item()

h_zymctrl = measure_resid_norm(plm_model, tokenizer, probe_seqs, TARGET_LAYER, hook_path="transformer.h")
matched_norm_1x = anchor_alpha_rel * h_zymctrl
v_matched_unit = v_raw * (matched_norm_1x / raw_norm)   # this IS "1x" in the matched sense

print(f"Layer {TARGET_LAYER}: raw utility-matched v_L norm = {raw_norm:.4f}")
print(f"ZymCTRL ‖h‖ at this layer (this run's probes): {h_zymctrl:.2f}")
print(f"matched norm for anchor_alpha_rel={anchor_alpha_rel:.4f} (this is '1x'): {matched_norm_1x:.4f}")
print(f"(38 measured 10.25 for this same quantity -- a close value confirms stability across runs)")


Reloading ZymCTRL on cuda to extract activations and measure ‖h‖...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: AI4PD/ZymCTRL
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layer 12: raw utility-matched v_L norm = 3.3572
ZymCTRL ‖h‖ at this layer (this run's probes): 51.26
matched norm for anchor_alpha_rel=0.2000 (this is '1x'): 10.2531
(38 measured 10.25 for this same quantity -- a close value confirms stability across runs)


In [7]:
# --- Generate: CONTROL + doses 1x-4x, all at the matched scale. ---

def generate_with_vector_steering(model, tokenizer, target_layer, steering_vector, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    model.eval()
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        return (out[0] + v,)

    records = []
    for prompt in prompts:
        handle = model.transformer.h[target_layer].register_forward_hook(hook)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, eos_token_id=EOS_ID, pad_token_id=PAD_ID
            )
        handle.remove()
        raw_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        seq = clean_zymctrl_output(raw_decoded)
        records.append({"prompt": prompt, "sequence": seq, "entropy": calculate_entropy(seq)})
    clear_gpu()
    return records

steer_prompts = build_ec_prompt_pool(EC_LABELS, len(DOSES) * 50, seed=2602)

conditions = {}
for i, dose in enumerate(DOSES):
    name = f"L12_MATCHED_{dose:g}x"
    lo, hi = i * 50, (i + 1) * 50
    vec = None if dose == 0.0 else v_matched_unit * dose
    print(f"=== {name} (prompts {lo}:{hi}, seed {1100+i}) ===")
    conditions[name] = generate_with_vector_steering(
        plm_model, tokenizer, TARGET_LAYER, vec, steer_prompts[lo:hi], seed=1100 + i)

print("\n=== Freeing ZymCTRL from GPU ===")
del plm_model
clear_gpu()


=== L12_MATCHED_0x (prompts 0:50, seed 1100) ===
=== L12_MATCHED_1x (prompts 50:100, seed 1101) ===
=== L12_MATCHED_2x (prompts 100:150, seed 1102) ===
=== L12_MATCHED_3x (prompts 150:200, seed 1103) ===
=== L12_MATCHED_4x (prompts 200:250, seed 1104) ===

=== Freeing ZymCTRL from GPU ===


In [8]:
# --- Fold with fold-success tracking. ---
evaluator2 = StructuralEvaluatorPTM()
for name in conditions:
    print(f"Folding {name}...")
    conditions[name] = fold_records_ptm(conditions[name], evaluator2)
del evaluator2
clear_gpu()

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

rows = []
for dose, name in zip(DOSES, conditions):
    recs = conditions[name]
    n = len(recs)
    n_ok = sum(1 for r in recs if r["fold_ok"])
    k = int(np.sum([r["collapse"] for r in recs]))
    lo, hi = wilson_ci(k, n)
    rows.append({
        "dose": dose, "condition": name, "collapsed": k, "n": n, "fold_ok": n_ok, "rate": k / n,
        "ci_lo": lo, "ci_hi": hi, "alpha_rel": dose * anchor_alpha_rel,
        "entropy": float(np.mean([r["entropy"] for r in recs])),
        "plddt": float(np.mean([r["plddt"] for r in recs if r["plddt"] > 0]) if n_ok else 0.0),
    })
dose_df = pd.DataFrame(rows)

print(f"\n{'Dose':>7s} {'alpha_rel':>10s} {'FoldOK':>7s} {'collapsed':>11s} {'rate':>8s} {'95% CI':>20s}")
print("-" * 78)
for _, r in dose_df.iterrows():
    ci_str = "[{:.1%}, {:.1%}]".format(r["ci_lo"], r["ci_hi"])
    print(f"{r['dose']:6.1f}x {r['alpha_rel']:10.4f} {int(r['fold_ok']):3d}/{int(r['n']):<3d} "
          f"{int(r['collapsed']):5d}/{int(r['n']):<5d} {r['rate']:7.1%} {ci_str:>20s}")


Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding L12_MATCHED_0x...
Folding L12_MATCHED_1x...
Folding L12_MATCHED_2x...
Folding L12_MATCHED_3x...
Folding L12_MATCHED_4x...

   Dose  alpha_rel  FoldOK   collapsed     rate               95% CI
------------------------------------------------------------------------------
   0.0x     0.0000  50/50     31/50      62.0%       [48.2%, 74.1%]
   1.0x     0.2000  50/50     34/50      68.0%       [54.2%, 79.2%]
   2.0x     0.4000  50/50     37/50      74.0%       [60.4%, 84.1%]
   3.0x     0.6000  50/50     43/50      86.0%       [73.8%, 93.0%]
   4.0x     0.8000  50/50     45/50      90.0%       [78.6%, 95.7%]


In [9]:
# --- Analysis: is there a threshold, and where does it sit relative to ProtGPT2's? ---
from scipy.stats import fisher_exact

ctrl_k = int(dose_df[dose_df["dose"] == 0.0]["collapsed"].iloc[0])
ctrl_n = int(dose_df[dose_df["dose"] == 0.0]["n"].iloc[0])

print("=" * 90)
print("PER-DOSE SIGNIFICANCE vs CONTROL")
print("=" * 90)
any_significant = False
first_significant_dose = None
for _, r in dose_df.iterrows():
    if r["dose"] == 0.0:
        continue
    k, n = int(r["collapsed"]), int(r["n"])
    _, p = fisher_exact([[k, n - k], [ctrl_k, ctrl_n - ctrl_k]])
    sig = p < 0.05
    if sig and first_significant_dose is None:
        first_significant_dose = r["dose"]
        any_significant = True
    print(f"  {r['dose']:.1f}x (alpha_rel={r['alpha_rel']:.4f}): p = {p:.4f}  "
          f"{'SIGNIFICANT' if sig else 'not significant'}")

print()
print("=" * 90)
print("VERDICT")
print("=" * 90)
if any_significant:
    print(f"ZymCTRL DOES show a real effect once pushed far enough -- first significant at "
          f"{first_significant_dose:.1f}x matched scale (alpha_rel = "
          f"{first_significant_dose * anchor_alpha_rel:.4f}).")
    print()
    print("This resolves the ambiguity from 38 in favor of explanation 2: ZymCTRL was not more")
    print("robust in kind, just needed a larger relative push than ProtGPT2 to show it. Compare")
    print(f"this threshold multiple ({first_significant_dose:.1f}x of the matched scale) to")
    print("ProtGPT2's own alpha_50=1.24x (locked-results.md SS1n) -- if meaningfully larger, that")
    print("is itself a real, reportable difference in HOW MUCH relative push each model tolerates,")
    print("even though both eventually show the same qualitative collapse. Report ZymCTRL's own")
    print("threshold multiple explicitly, not just 'it also collapses eventually'.")
else:
    print("ZymCTRL shows NO significant effect even at 4x the matched scale -- roughly double the")
    print("relative push ProtGPT2 needed to reach ~90%+ collapse (alpha_50=1.24x, locked SS1n).")
    print()
    print("This is now real evidence for explanation 1: ZymCTRL appears genuinely more robust to")
    print("this class of perturbation, not just requiring a bigger push to show the same pattern.")
    print("Report this as a positive, specific finding -- EC-conditioning (or some other property")
    print("of this architecture/training) appears to buy real robustness to activation steering,")
    print("worth a sentence of speculation on mechanism (tighter output distribution -> more")
    print("redundant encoding of structural information) explicitly flagged as speculation, not")
    print("verified. A genuinely interesting, non-obvious result for the paper either way.")

print()
print("Read against 38's original 1x/2x (same construction, this run's fresh measurements should")
print("closely match):")
print("  38: CONTROL 66.0% -> 1x 74.0% (p=0.51) -> 2x 72.0% (p=0.67)")
c1 = dose_df[dose_df["dose"] == 1.0]
c2 = dose_df[dose_df["dose"] == 2.0]
print(f"  this run: CONTROL {dose_df[dose_df['dose']==0.0]['rate'].iloc[0]:.1%} -> "
      f"1x {c1['rate'].iloc[0]:.1%} -> 2x {c2['rate'].iloc[0]:.1%}")


PER-DOSE SIGNIFICANCE vs CONTROL
  1.0x (alpha_rel=0.2000): p = 0.6753  not significant
  2.0x (alpha_rel=0.4000): p = 0.2837  not significant
  3.0x (alpha_rel=0.6000): p = 0.0113  SIGNIFICANT
  4.0x (alpha_rel=0.8000): p = 0.0019  SIGNIFICANT

VERDICT
ZymCTRL DOES show a real effect once pushed far enough -- first significant at 3.0x matched scale (alpha_rel = 0.6000).

This resolves the ambiguity from 38 in favor of explanation 2: ZymCTRL was not more
robust in kind, just needed a larger relative push than ProtGPT2 to show it. Compare
this threshold multiple (3.0x of the matched scale) to
ProtGPT2's own alpha_50=1.24x (locked-results.md SS1n) -- if meaningfully larger, that
is itself a real, reportable difference in HOW MUCH relative push each model tolerates,
even though both eventually show the same qualitative collapse. Report ZymCTRL's own
threshold multiple explicitly, not just 'it also collapses eventually'.

Read against 38's original 1x/2x (same construction, this run's fres

In [10]:
# --- Persist. ---
seq_rows = []
for dose, name in zip(DOSES, conditions):
    for i, r in enumerate(conditions[name]):
        seq_rows.append({
            "condition": name, "dose": dose, "alpha_rel": dose * anchor_alpha_rel, "idx": i,
            "prompt": r["prompt"], "sequence": r["sequence"], "gen_length": len(r["sequence"]),
            "usable_length": sum(1 for a in r["sequence"] if a in VALID_AA),
            "entropy": r["entropy"], "plddt": r["plddt"], "ptm": r["ptm"],
            "fold_ok": r["fold_ok"], "collapse": r["collapse"],
        })
pd.DataFrame(seq_rows).to_csv("zymctrl_threshold_search_sequences.csv", index=False)
dose_df.to_csv("zymctrl_threshold_search_summary.csv", index=False)

pd.DataFrame([{
    "anchor_alpha_rel": anchor_alpha_rel, "h_protgpt2_l12": h_protgpt2_l12,
    "h_zymctrl_l12": h_zymctrl, "matched_norm_1x": matched_norm_1x,
}]).to_csv("zymctrl_threshold_search_calibration.csv", index=False)

print("Saved:")
print("  zymctrl_threshold_search_sequences.csv")
print("  zymctrl_threshold_search_summary.csv")
print("  zymctrl_threshold_search_calibration.csv")
print()
print("Update notes/locked-results.md SS1h-REPAIRED and notes/context-and-decisions.md SS10.7")
print("with this run's verdict -- it resolves the 'genuinely robust vs. under-pushed' question")
print("that SS10.7 explicitly left open.")


Saved:
  zymctrl_threshold_search_sequences.csv
  zymctrl_threshold_search_summary.csv
  zymctrl_threshold_search_calibration.csv

Update notes/locked-results.md SS1h-REPAIRED and notes/context-and-decisions.md SS10.7
with this run's verdict -- it resolves the 'genuinely robust vs. under-pushed' question
that SS10.7 explicitly left open.
